# Fully Compressible 2D Fluid Solver (Collocated Grid)
## IDC 606 — High Performance Computing

**Numerical scheme:** MUSCL reconstruction + HLLC Riemann solver + TVD-RK3 time stepping  
**Verification:** Sod Shock Tube (exact solution comparison)  
**Deliverables:** Schlieren plot · Mach number map · Energy conservation · 2-D explosion

## 1. Parameters

In [ ]:
%%writefile para.py
import numpy as np

device = "CPU"

# Time
tinit  = 0.0
tfinal = 0.2          # long enough to see Sod shock develop
dt     = 1e-4         # initial guess; overridden by CFL each step

# Gas properties
gamma = 1.4
R     = 287.0

# Grid
Nx = 200              # higher res for Sod shock tube
Ny = 4                # thin in y → effectively 1-D for Sod test
Lx = 1.0
Ly = 0.02             # thin slab

# Output
t_print = 0.04        # print interval
output_dir = "output"


## 2. Mesh and Grid

In [ ]:
%%writefile mesh_and_grid.py
import numpy as np
import para

# --------------------------------------------------
# GRID DIMENSIONS
# --------------------------------------------------
Nx = para.Nx
Ny = para.Ny
Lx = para.Lx
Ly = para.Ly

dx = Lx / Nx
dy = Ly / Ny

# --------------------------------------------------
# CELL-CENTERED COORDINATES
# --------------------------------------------------
x_centers = (np.arange(Nx) + 0.5) * dx
y_centers = (np.arange(Ny) + 0.5) * dy

X_mesh, Y_mesh = np.meshgrid(x_centers, y_centers, indexing='ij')

print(f"[mesh] Grid: {Nx} x {Ny}   dx={dx:.4f}  dy={dy:.4f}")


## 3. Compressible State (Conserved / Primitive Variables)

In [ ]:
%%writefile compressible.py
import numpy as np
import mesh_and_grid as grid
import para

gamma = para.gamma

# --------------------------------------------------
# PRIMITIVE VARIABLES  (shape: Nx x Ny)
# --------------------------------------------------
rho = np.ones  ((grid.Nx, grid.Ny))
ux  = np.zeros ((grid.Nx, grid.Ny))
uy  = np.zeros ((grid.Nx, grid.Ny))
p   = np.ones  ((grid.Nx, grid.Ny))

# --------------------------------------------------
# CONSERVED VARIABLES  Q = [rho, rho*u, rho*v, E]
# --------------------------------------------------
Q = np.zeros((4, grid.Nx, grid.Ny))


# --------------------------------------------------
# PRIMITIVE → CONSERVED
# --------------------------------------------------
def prim_to_cons():
    """Fill Q from (rho, ux, uy, p)."""
    Q[0] = rho
    Q[1] = rho * ux
    Q[2] = rho * uy
    E    = p / ((gamma - 1.0)) + 0.5 * rho * (ux**2 + uy**2)
    Q[3] = E


# --------------------------------------------------
# CONSERVED → PRIMITIVE
# --------------------------------------------------
def cons_to_prim():
    """Fill (rho, ux, uy, p) from Q."""
    rho[:] = np.maximum(Q[0], 1e-10)
    ux [:]  = Q[1] / rho
    uy [:]  = Q[2] / rho
    e_int   = Q[3] / rho - 0.5 * (ux**2 + uy**2)
    p  [:]  = np.maximum((gamma - 1.0) * rho * e_int, 1e-10)


# --------------------------------------------------
# PHYSICAL FLUX VECTORS  (return arrays, not derivatives)
# --------------------------------------------------
def flux_x(Qv):
    """
    Given conserved state Qv (4,Nx,Ny), return physical flux F_x (4,Nx,Ny).
    """
    r  = np.maximum(Qv[0], 1e-10)
    ru = Qv[1];  rv = Qv[2];  E = Qv[3]
    u  = ru / r
    v  = rv / r
    e  = E / r - 0.5 * (u**2 + v**2)
    pp = np.maximum((gamma - 1.0) * r * e, 1e-10)

    F = np.empty_like(Qv)
    F[0] = ru
    F[1] = ru * u + pp
    F[2] = ru * v
    F[3] = u * (E + pp)
    return F


def flux_y(Qv):
    """
    Given conserved state Qv (4,Nx,Ny), return physical flux F_y (4,Nx,Ny).
    """
    r  = np.maximum(Qv[0], 1e-10)
    ru = Qv[1];  rv = Qv[2];  E = Qv[3]
    u  = ru / r
    v  = rv / r
    e  = E / r - 0.5 * (u**2 + v**2)
    pp = np.maximum((gamma - 1.0) * r * e, 1e-10)

    G = np.empty_like(Qv)
    G[0] = rv
    G[1] = rv * u
    G[2] = rv * v + pp
    G[3] = v * (E + pp)
    return G


## 4. MUSCL Reconstruction (Slope Limiter)

In [ ]:
%%writefile reconstruction.py
"""
reconstruction.py
-----------------
MUSCL linear reconstruction with minmod slope limiter.

For each interface i+1/2 we produce:
    Q_L[i]   = left  state (from cell i,   extrapolated rightward)
    Q_R[i]   = right state (from cell i+1, extrapolated leftward)

Indices convention:
    Q        shape (4, Nx, Ny)
    Q_L, Q_R shape (4, Nx+1, Ny)   — one extra interface on each side
"""

import numpy as np


# --------------------------------------------------
# MINMOD LIMITER
# --------------------------------------------------
def minmod(a, b):
    return 0.5 * (np.sign(a) + np.sign(b)) * np.minimum(np.abs(a), np.abs(b))


# --------------------------------------------------
# MUSCL RECONSTRUCTION IN X
# --------------------------------------------------
def reconstruct_x(Q):
    """
    Returns Q_L, Q_R of shape (4, Nx+1, Ny).
    Interface k sits between cell k-1 and cell k  (k = 0..Nx).
    """
    k, Nx, Ny = Q.shape

    # Extend Q with one ghost cell on each side (periodic)
    Qg = np.concatenate([Q[:, -1:, :], Q, Q[:, :1, :]], axis=1)  # (4, Nx+2, Ny)

    # Slopes inside each physical cell  (size Nx)
    dQ_L = Qg[:, 1:-1, :] - Qg[:, :-2, :]   # Q[i] - Q[i-1]
    dQ_R = Qg[:, 2:,   :] - Qg[:, 1:-1, :]  # Q[i+1] - Q[i]
    slope = minmod(dQ_L, dQ_R)               # (4, Nx, Ny)

    # Extrapolate to right face of cell i  →  left state of interface i+1/2
    # Extrapolate to left  face of cell i  →  right state of interface i-1/2
    Q_face_R = Q + 0.5 * slope   # right face of cell i
    Q_face_L = Q - 0.5 * slope   # left  face of cell i

    # Interface k (between cell k-1 and k):
    #   left  state = right face of cell k-1
    #   right state = left  face of cell k
    # We need interfaces k=0..Nx  (Nx+1 total)

    # Pad to get interfaces at boundaries (periodic)
    Q_L = np.concatenate([Q_face_R[:, -1:, :], Q_face_R], axis=1)   # (4, Nx+1, Ny)
    Q_R = np.concatenate([Q_face_L,             Q_face_L[:, :1, :]], axis=1)   # (4, Nx+1, Ny)

    return Q_L, Q_R


# --------------------------------------------------
# MUSCL RECONSTRUCTION IN Y
# --------------------------------------------------
def reconstruct_y(Q):
    """
    Returns Q_L, Q_R of shape (4, Nx, Ny+1).
    Interface k sits between cell k-1 and cell k  (k = 0..Ny).
    """
    k, Nx, Ny = Q.shape

    Qg = np.concatenate([Q[:, :, -1:], Q, Q[:, :, :1]], axis=2)   # (4, Nx, Ny+2)

    dQ_L = Qg[:, :, 1:-1] - Qg[:, :, :-2]
    dQ_R = Qg[:, :, 2:  ] - Qg[:, :, 1:-1]
    slope = minmod(dQ_L, dQ_R)                                       # (4, Nx, Ny)

    Q_face_R = Q + 0.5 * slope
    Q_face_L = Q - 0.5 * slope

    Q_L = np.concatenate([Q_face_R[:, :, -1:], Q_face_R], axis=2)  # (4, Nx, Ny+1)
    Q_R = np.concatenate([Q_face_L,             Q_face_L[:, :, :1]], axis=2)

    return Q_L, Q_R


## 5. HLLC Riemann Solver

In [ ]:
%%writefile riemann.py
"""
riemann.py
----------
HLLC approximate Riemann solver for the 2D Euler equations.

References:
    Toro (2009) "Riemann Solvers and Numerical Methods for Fluid Dynamics"
    Chapter 10 — HLLC solver.

hllc_flux_x(QL, QR, gamma)
    QL, QR : shape (4, M, N)  — left / right states at M interfaces
    returns F_hllc : shape (4, M, N)

hllc_flux_y(QL, QR, gamma)
    Rotates the problem: normal direction becomes y.
"""

import numpy as np


def _prim_from_cons(Qv, gamma):
    """Extract primitives from conserved state array (4, ...)."""
    r  = np.maximum(Qv[0], 1e-10)
    u  = Qv[1] / r
    v  = Qv[2] / r
    e  = Qv[3] / r - 0.5 * (u**2 + v**2)
    pp = np.maximum((gamma - 1.0) * r * e, 1e-10)
    a  = np.sqrt(gamma * pp / r)
    return r, u, v, pp, a


def hllc_flux_x(QL, QR, gamma=1.4):
    """
    HLLC flux in the x-direction.
    QL, QR : (4, ...) conserved state arrays.
    Returns interface flux (4, ...).
    """
    rL, uL, vL, pL, aL = _prim_from_cons(QL, gamma)
    rR, uR, vR, pR, aR = _prim_from_cons(QR, gamma)

    # --------------------------------------------------
    # 1) Wave speed estimates  (Einfeldt / Roe average)
    # --------------------------------------------------
    SL = np.minimum(uL - aL,  uR - aR)   # left-most  wave
    SR = np.maximum(uL + aL,  uR + aR)   # right-most wave

    # Contact wave speed  S* (Toro 10.37)
    num   = pR - pL + rL * uL * (SL - uL) - rR * uR * (SR - uR)
    denom = rL * (SL - uL)              - rR      * (SR - uR)
    denom = np.where(np.abs(denom) < 1e-14, 1e-14, denom)
    Sstar = num / denom

    # --------------------------------------------------
    # 2) Physical fluxes at left and right states
    # --------------------------------------------------
    EL = QL[3];   ER = QR[3]

    FL = np.empty_like(QL)
    FL[0] = rL * uL
    FL[1] = rL * uL**2 + pL
    FL[2] = rL * uL * vL
    FL[3] = uL * (EL + pL)

    FR = np.empty_like(QR)
    FR[0] = rR * uR
    FR[1] = rR * uR**2 + pR
    FR[2] = rR * uR * vR
    FR[3] = uR * (ER + pR)

    # --------------------------------------------------
    # 3) HLLC star states  (Toro 10.38 - 10.40)
    # --------------------------------------------------
    def star_state(Q, r, u, v, p, E, S, Ss):
        coeff = r * (S - u) / (S - Ss)
        Qs    = np.empty_like(Q)
        Qs[0] = coeff
        Qs[1] = coeff * Ss
        Qs[2] = coeff * v
        Qs[3] = coeff * (E / r + (Ss - u) * (Ss + p / (r * (S - u))))
        return Qs

    QL_star = star_state(QL, rL, uL, vL, pL, EL, SL, Sstar)
    QR_star = star_state(QR, rR, uR, vR, pR, ER, SR, Sstar)

    # HLLC flux (Toro 10.26)
    FL_star = FL + SL * (QL_star - QL)
    FR_star = FR + SR * (QR_star - QR)

    # --------------------------------------------------
    # 4) Select correct region
    # --------------------------------------------------
    F = np.where(SL[np.newaxis, ...]   >= 0,  FL,
        np.where(Sstar[np.newaxis, ...] >= 0,  FL_star,
        np.where(SR[np.newaxis, ...]   >= 0,   FR_star,
                                                FR)))
    return F


def hllc_flux_y(QL, QR, gamma=1.4):
    """
    HLLC flux in the y-direction.
    Swap u <-> v components to reuse the x-flux routine.
    """
    def swap_uv(Qv):
        Qs = Qv.copy()
        Qs[1] = Qv[2]   # rho*v → normal momentum
        Qs[2] = Qv[1]   # rho*u → tangential momentum
        return Qs

    QL_r = swap_uv(QL)
    QR_r = swap_uv(QR)

    F_r = hllc_flux_x(QL_r, QR_r, gamma)

    # Swap back
    F = F_r.copy()
    F[1] = F_r[2]
    F[2] = F_r[1]
    return F


## 6. Flux Divergence Operator

In [ ]:
%%writefile flux_divergence.py
"""
flux_divergence.py
------------------
Combines MUSCL reconstruction and HLLC Riemann solver to produce
the net flux divergence dQ/dt = -( dF/dx + dG/dy ).

Uses periodic boundary conditions (wrapped in reconstruction.py).
"""

import numpy as np
import para
from reconstruction import reconstruct_x, reconstruct_y
from riemann        import hllc_flux_x, hllc_flux_y

gamma = para.gamma


def compute_rhs(Q, dx, dy):
    """
    Compute RHS = -(dF/dx + dG/dy)  for the 2D Euler equations.

    Parameters
    ----------
    Q  : (4, Nx, Ny) conserved state
    dx : cell width in x
    dy : cell width in y

    Returns
    -------
    rhs : (4, Nx, Ny)
    """
    Nx = Q.shape[1]
    Ny = Q.shape[2]

    # ---- X-direction -----------------------------------------------
    # reconstruct_x returns (4, Nx+1, Ny) interface states
    QL_x, QR_x = reconstruct_x(Q)            # (4, Nx+1, Ny)

    # Compute HLLC flux at every interface
    Fx = hllc_flux_x(QL_x, QR_x, gamma)      # (4, Nx+1, Ny)

    # Net flux divergence: (F_{i+1/2} - F_{i-1/2}) / dx
    #   Fx[:, 1:Nx+1, :] = flux at right face of cell i
    #   Fx[:, 0:Nx,   :] = flux at left  face of cell i
    dFx = (Fx[:, 1:Nx+1, :] - Fx[:, 0:Nx, :]) / dx   # (4, Nx, Ny)

    # ---- Y-direction -----------------------------------------------
    QL_y, QR_y = reconstruct_y(Q)            # (4, Nx, Ny+1)

    Fy = hllc_flux_y(QL_y, QR_y, gamma)      # (4, Nx, Ny+1)

    dFy = (Fy[:, :, 1:Ny+1] - Fy[:, :, 0:Ny]) / dy   # (4, Nx, Ny)

    return -(dFx + dFy)


## 7. Time Integration (CFL + TVD-RK3)

In [ ]:
%%writefile fns.py
"""
fns.py
------
Time integration for the compressible Euler solver.

Provides:
    compute_dt(Q, dx, dy, CFL)   → adaptive time step
    time_advance_rk3(Q, dt, dx, dy) → one RK3 step
    run_simulation(Q, ...)       → main loop
"""

import numpy as np
import para
from flux_divergence import compute_rhs

gamma = para.gamma


# --------------------------------------------------
# ADAPTIVE CFL TIME STEP
# --------------------------------------------------
def compute_dt(Q, dx, dy, CFL=0.45):
    """
    Compute stable dt from the CFL condition.

    dt = CFL * min(dx, dy) / max_wave_speed
    """
    r  = np.maximum(Q[0], 1e-10)
    u  = Q[1] / r
    v  = Q[2] / r
    e  = Q[3] / r - 0.5 * (u**2 + v**2)
    pp = np.maximum((gamma - 1.0) * r * e, 1e-10)
    a  = np.sqrt(gamma * pp / r)

    max_speed_x = np.max(np.abs(u) + a)
    max_speed_y = np.max(np.abs(v) + a)

    max_speed_x = max(max_speed_x, 1e-10)
    max_speed_y = max(max_speed_y, 1e-10)

    dt = CFL * min(dx / max_speed_x, dy / max_speed_y)
    return dt


# --------------------------------------------------
# TVD RUNGE-KUTTA 3  (Shu-Osher, 1988)
# --------------------------------------------------
def time_advance_rk3(Q, dt, dx, dy):
    """
    Advance Q by one time step using 3rd-order TVD RK3.

    Shu-Osher coefficients:
        Q1 = Q  + dt * L(Q)
        Q2 = (3/4)*Q + (1/4)*(Q1 + dt*L(Q1))
        Q3 = (1/3)*Q + (2/3)*(Q2 + dt*L(Q2))
    """
    L0 = compute_rhs(Q,  dx, dy)
    Q1 = Q + dt * L0

    L1 = compute_rhs(Q1, dx, dy)
    Q2 = 0.75 * Q + 0.25 * (Q1 + dt * L1)

    L2 = compute_rhs(Q2, dx, dy)
    Q3 = (1.0/3.0) * Q + (2.0/3.0) * (Q2 + dt * L2)

    return Q3


# --------------------------------------------------
# MAIN SIMULATION LOOP
# --------------------------------------------------
def run_simulation(Q, dx, dy, tfinal, t_print=0.04, CFL=0.45):
    """
    Run the simulation from t=0 to t=tfinal.

    Parameters
    ----------
    Q        : initial conserved state (4, Nx, Ny)
    dx, dy   : grid spacing
    tfinal   : end time
    t_print  : how often to print status
    CFL      : Courant number

    Returns
    -------
    Q        : final state
    history  : list of (t, Q_snapshot) tuples saved every t_print
    """
    t       = 0.0
    step    = 0
    t_next  = t_print
    history = []

    print(f"{'Step':>6}  {'t':>9}  {'dt':>10}  {'rho_mean':>10}  {'p_mean':>10}")
    print("-" * 55)

    while t < tfinal:
        dt = compute_dt(Q, dx, dy, CFL)
        dt = min(dt, tfinal - t)   # don't overshoot

        Q  = time_advance_rk3(Q, dt, dx, dy)

        # Clip to prevent negative density / pressure blow-up
        Q[0] = np.maximum(Q[0], 1e-10)

        t    += dt
        step += 1

        if t >= t_next or step % 200 == 0:
            r  = np.maximum(Q[0], 1e-10)
            u  = Q[1] / r
            v  = Q[2] / r
            e  = Q[3] / r - 0.5 * (u**2 + v**2)
            pp = np.maximum((gamma - 1.0) * r * e, 1e-10)
            print(f"{step:>6}  {t:>9.5f}  {dt:>10.3e}  "
                  f"{r.mean():>10.4f}  {pp.mean():>10.4f}")
            history.append((t, Q.copy()))
            t_next += t_print

        if np.any(~np.isfinite(Q)):
            print("⚠ NaN / Inf detected — stopping.")
            break

    print(f"\nDone  t={t:.5f}  steps={step}")
    return Q, history


## 8. Initial Conditions

In [ ]:
%%writefile init_fields.py
"""
init_fields.py
--------------
Initial condition library for the compressible 2D Euler solver.

Available routines:
    init_sod(Q, X)         — Sod shock tube  (primary verification test)
    init_uniform(Q, X)     — uniform Mach-N flow
    init_kh(Q, X, Y)       — Kelvin-Helmholtz shear layer
    init_explosion(Q,X,Y)  — circular blast wave (2-D test)
"""

import numpy as np
import para

gamma = para.gamma


# --------------------------------------------------
# 1. SOD SHOCK TUBE
# --------------------------------------------------
def init_sod(Q, X, Y):
    """
    Classic Sod shock tube along x.
    Left:  rho=1.0, u=0, p=1.0
    Right: rho=0.125, u=0, p=0.1
    Diaphragm at x=0.5.
    """
    Nx, Ny = X.shape

    rho = np.where(X < 0.5, 1.0,   0.125)
    ux  = np.zeros_like(X)
    uy  = np.zeros_like(X)
    pp  = np.where(X < 0.5, 1.0,   0.1)

    Q[0] = rho
    Q[1] = rho * ux
    Q[2] = rho * uy
    Q[3] = pp / (gamma - 1.0) + 0.5 * rho * (ux**2 + uy**2)

    print("[init] Sod shock tube  — diaphragm at x=0.5")
    print(f"       rho range [{rho.min():.3f}, {rho.max():.3f}]   "
          f"p range [{pp.min():.3f}, {pp.max():.3f}]")


# --------------------------------------------------
# 2. UNIFORM MACH FLOW  (your original init)
# --------------------------------------------------
def init_uniform(Q, X, Y, Mach=2.0, T0=300.0, p0=101325.0, R=287.0):
    """
    Uniform flow at given Mach number with small pressure perturbation.
    """
    a0   = np.sqrt(gamma * R * T0)
    u0   = Mach * a0
    rho0 = p0 / (R * T0)

    rho = rho0 * np.ones_like(X)
    ux  = u0   * np.ones_like(X)
    uy  = np.zeros_like(X)
    pp  = p0   * np.ones_like(X)

    # Small perturbation
    pp += 1e-3 * p0 * np.sin(2 * np.pi * X) * np.sin(2 * np.pi * Y)

    Q[0] = rho
    Q[1] = rho * ux
    Q[2] = rho * uy
    Q[3] = pp / (gamma - 1.0) + 0.5 * rho * (ux**2 + uy**2)

    print(f"[init] Uniform Mach={Mach:.1f}  rho={rho0:.3f}  p={p0:.1f}")


# --------------------------------------------------
# 3. KELVIN-HELMHOLTZ SHEAR LAYER
# --------------------------------------------------
def init_kh(Q, X, Y, Mach=0.5):
    """
    Two counter-streaming layers for Kelvin-Helmholtz instability.
    Domain should be [0,1]x[0,1] periodic.
    """
    rho = np.ones_like(X)
    pp  = (1.0 / gamma) * np.ones_like(X)   # p = 1/gamma → M~1 at u=1
    a0  = np.sqrt(gamma * pp / rho)
    u0  = Mach * a0

    ux  = np.where(Y < 0.5, u0, -u0)

    # Random perturbation in vy to seed the instability
    np.random.seed(42)
    uy  = 1e-2 * np.random.randn(*X.shape)

    Q[0] = rho
    Q[1] = rho * ux
    Q[2] = rho * uy
    Q[3] = pp / (gamma - 1.0) + 0.5 * rho * (ux**2 + uy**2)

    print(f"[init] Kelvin-Helmholtz  Mach={Mach:.2f}")


# --------------------------------------------------
# 4. CIRCULAR EXPLOSION  (2-D blast wave)
# --------------------------------------------------
def init_explosion(Q, X, Y, r_blast=0.2, p_in=10.0, p_out=1.0, rho0=1.0):
    """
    Circular high-pressure region in the centre of the domain.
    """
    xc = X.mean();  yc = Y.mean()
    r2 = (X - xc)**2 + (Y - yc)**2

    rho = rho0 * np.ones_like(X)
    ux  = np.zeros_like(X)
    uy  = np.zeros_like(X)
    pp  = np.where(r2 < r_blast**2, p_in, p_out)

    Q[0] = rho
    Q[1] = rho * ux
    Q[2] = rho * uy
    Q[3] = pp / (gamma - 1.0) + 0.5 * rho * (ux**2 + uy**2)

    print(f"[init] Circular explosion  r_blast={r_blast}  "
          f"p_in={p_in}  p_out={p_out}")


## 9. Exact Sod Solution (Verification Reference)

In [ ]:
%%writefile sod_exact.py
"""
sod_exact.py
------------
Exact solution for the Sod shock tube problem.

Left state:  rho=1.0, u=0, p=1.0
Right state: rho=0.125, u=0, p=0.1
Diaphragm at x=0.5, gamma=1.4

Usage:
    from sod_exact import sod_solution
    rho_ex, u_ex, p_ex = sod_solution(x_array, t)
"""

import numpy as np


def sod_solution(x, t, gamma=1.4, x0=0.5):
    """
    Returns exact (rho, u, p) arrays at positions x and time t.
    Iteratively solves for the contact/shock speeds.
    """
    # Initial states
    rhoL, uL, pL = 1.0,   0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1

    aL = np.sqrt(gamma * pL / rhoL)
    aR = np.sqrt(gamma * pR / rhoR)

    # ----------------------------------------------------------------
    # Find p_star via Newton iteration (Toro Ch4)
    # ----------------------------------------------------------------
    def fL(p):
        if p > pL:
            A = 2.0 / ((gamma + 1) * rhoL)
            B = (gamma - 1) / (gamma + 1) * pL
            return (p - pL) * np.sqrt(A / (p + B))
        else:
            return (2 * aL / (gamma - 1)) * ((p / pL)**((gamma - 1) / (2 * gamma)) - 1)

    def fR(p):
        if p > pR:
            A = 2.0 / ((gamma + 1) * rhoR)
            B = (gamma - 1) / (gamma + 1) * pR
            return (p - pR) * np.sqrt(A / (p + B))
        else:
            return (2 * aR / (gamma - 1)) * ((p / pR)**((gamma - 1) / (2 * gamma)) - 1)

    def f(p):
        return fL(p) + fR(p) + (uR - uL)

    def df(p):
        dp = 1e-6 * p
        return (f(p + dp) - f(p - dp)) / (2 * dp)

    p_star = 0.5 * (pL + pR)
    for _ in range(100):
        dp = -f(p_star) / df(p_star)
        p_star += dp
        if abs(dp) < 1e-12 * p_star:
            break

    u_star = 0.5 * (uL + uR) + 0.5 * (fR(p_star) - fL(p_star))

    # ----------------------------------------------------------------
    # Wave speeds
    # ----------------------------------------------------------------
    # Rarefaction (left)
    aL_star = aL * (p_star / pL)**((gamma - 1) / (2 * gamma))
    S_HL    = uL - aL          # head of rarefaction
    S_TL    = u_star - aL_star # tail of rarefaction

    # Contact discontinuity
    S_contact = u_star

    # Shock (right)
    S_R = uR + aR * np.sqrt((gamma + 1) / (2 * gamma) * (p_star / pR) +
                              (gamma - 1) / (2 * gamma))

    # Densities in the star regions
    rhoL_star = rhoL * (p_star / pL)**(1 / gamma)
    rhoR_star = rhoR * ((p_star / pR + (gamma - 1) / (gamma + 1)) /
                        ((gamma - 1) / (gamma + 1) * p_star / pR + 1))

    # ----------------------------------------------------------------
    # Sample solution  xi = (x - x0) / t
    # ----------------------------------------------------------------
    if t <= 0:
        rho = np.where(x < x0, rhoL, rhoR)
        u   = np.where(x < x0, uL,   uR)
        p   = np.where(x < x0, pL,   pR)
        return rho, u, p

    xi = (x - x0) / t

    rho = np.empty_like(x, dtype=float)
    u_f = np.empty_like(x, dtype=float)
    pp  = np.empty_like(x, dtype=float)

    for i, s in enumerate(xi):
        if s <= S_HL:                                # left undisturbed
            rho[i] = rhoL;  u_f[i] = uL;  pp[i] = pL
        elif s <= S_TL:                              # inside rarefaction
            u_tmp   = 2 / (gamma + 1) * (aL + (gamma - 1) / 2 * uL + s)
            a_tmp   = aL + (gamma - 1) / 2 * (uL - u_tmp)
            rho[i]  = rhoL * (a_tmp / aL)**(2 / (gamma - 1))
            u_f[i]  = u_tmp
            pp[i]   = pL * (a_tmp / aL)**(2 * gamma / (gamma - 1))
        elif s <= S_contact:                         # left star region
            rho[i] = rhoL_star;  u_f[i] = u_star;  pp[i] = p_star
        elif s <= S_R:                               # right star region
            rho[i] = rhoR_star;  u_f[i] = u_star;  pp[i] = p_star
        else:                                        # right undisturbed
            rho[i] = rhoR;  u_f[i] = uR;  pp[i] = pR

    return rho, u_f, pp


## 10. Plotting and Saving

In [ ]:
%%writefile saving.py
"""
saving.py
---------
All plotting and saving routines for the compressible solver.

Functions:
    plot_sod_verification(Q, X, t, gamma)     — compare with exact Sod solution
    plot_contours(Q, X, Y, t, gamma)          — rho, u, p, Mach contours
    plot_schlieren(Q, X, Y, dx, dy, t)        — Schlieren (|grad rho|)
    plot_mach(Q, X, Y, t, gamma)              — Mach number map
    plot_energy_history(history, dx, dy, gamma) — total energy over time
    save_all(Q, X, Y, dx, dy, t, gamma, tag)  — save all plots at once
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

import para

gamma = para.gamma
os.makedirs("output", exist_ok=True)


# --------------------------------------------------
# HELPERS
# --------------------------------------------------
def _primitives(Q):
    """Extract (rho, ux, uy, p, a, Mach) from conserved state."""
    r  = np.maximum(Q[0], 1e-10)
    u  = Q[1] / r
    v  = Q[2] / r
    e  = Q[3] / r - 0.5 * (u**2 + v**2)
    pp = np.maximum((gamma - 1.0) * r * e, 1e-10)
    a  = np.sqrt(gamma * pp / r)
    M  = np.sqrt(u**2 + v**2) / a
    return r, u, v, pp, a, M


# --------------------------------------------------
# 1. SOD VERIFICATION PLOT
# --------------------------------------------------
def plot_sod_verification(Q, X, t, save=True):
    """
    Compare numerical 1-D Sod solution (averaged over y) with the exact answer.
    """
    from sod_exact import sod_solution

    r, u, v, pp, a, M = _primitives(Q)

    # Average over y to get 1-D profiles
    x1d   = X[:, 0]
    rho1d = r.mean(axis=1)
    u1d   = u.mean(axis=1)
    p1d   = pp.mean(axis=1)

    # Exact solution
    rho_ex, u_ex, p_ex = sod_solution(x1d, t)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(f"Sod Shock Tube Verification   t = {t:.4f}", fontsize=13)

    pairs = [
        (axes[0], rho1d, rho_ex, "Density  ρ",       "tab:blue"),
        (axes[1], u1d,   u_ex,   "Velocity  u",      "tab:green"),
        (axes[2], p1d,   p_ex,   "Pressure  p",      "tab:red"),
    ]

    for ax, num, ex, title, col in pairs:
        ax.plot(x1d, ex,  'k-',  lw=2,   label="Exact",     zorder=3)
        ax.plot(x1d, num, 'o--', color=col, ms=3, lw=1.2,
                label="MUSCL-HLLC", zorder=2)
        ax.set_title(title);  ax.set_xlabel("x")
        ax.legend(fontsize=8);  ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if save:
        fname = f"output/sod_verification_t{t:.4f}.png"
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f"  Saved → {fname}")
    plt.show()


# --------------------------------------------------
# 2. FIELD CONTOURS  (rho, ux, p, Mach)
# --------------------------------------------------
def plot_contours(Q, X, Y, t, save=True):
    r, u, v, pp, a, M = _primitives(Q)

    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    fig.suptitle(f"Flow Field   t = {t:.4f}", fontsize=13)

    fields = [
        (axes[0, 0], r,  "Density  ρ",      "viridis"),
        (axes[0, 1], u,  "Velocity  uₓ",    "RdBu_r"),
        (axes[1, 0], pp, "Pressure  p",      "plasma"),
        (axes[1, 1], M,  "Mach Number",      "hot"),
    ]

    for ax, field, title, cmap in fields:
        cf = ax.contourf(X, Y, field, levels=40, cmap=cmap)
        plt.colorbar(cf, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(title);  ax.set_xlabel("x");  ax.set_ylabel("y")
        ax.set_aspect('auto')

    plt.tight_layout()
    if save:
        fname = f"output/contours_t{t:.4f}.png"
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f"  Saved → {fname}")
    plt.show()


# --------------------------------------------------
# 3. SCHLIEREN PLOT  ( exp(-k |∇ρ|) )
# --------------------------------------------------
def plot_schlieren(Q, X, Y, dx, dy, t, k=10.0, save=True):
    """
    Numerical Schlieren: s = exp(-k * |grad(rho)| / max|grad(rho)|)
    Dark regions = high density gradient = shocks / contact surfaces.
    """
    r = _primitives(Q)[0]

    drho_dx = np.gradient(r, dx, axis=0)
    drho_dy = np.gradient(r, dy, axis=1)
    grad_mag = np.sqrt(drho_dx**2 + drho_dy**2)
    grad_norm = grad_mag / (grad_mag.max() + 1e-14)

    schlieren = np.exp(-k * grad_norm)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.imshow(schlieren.T, origin='lower',
              extent=[X.min(), X.max(), Y.min(), Y.max()],
              cmap='gray', vmin=0, vmax=1)
    ax.set_title(f"Numerical Schlieren  |∇ρ|   t = {t:.4f}")
    ax.set_xlabel("x");  ax.set_ylabel("y")

    plt.tight_layout()
    if save:
        fname = f"output/schlieren_t{t:.4f}.png"
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f"  Saved → {fname}")
    plt.show()


# --------------------------------------------------
# 4. MACH NUMBER MAP
# --------------------------------------------------
def plot_mach(Q, X, Y, t, save=True):
    M = _primitives(Q)[5]

    fig, ax = plt.subplots(figsize=(7, 5))
    cf = ax.contourf(X, Y, M, levels=40, cmap='jet')
    cb = plt.colorbar(cf, ax=ax)
    cb.set_label("Mach Number")

    # Highlight sonic line M=1
    ax.contour(X, Y, M, levels=[1.0], colors='white', linewidths=1.5,
               linestyles='--')

    ax.set_title(f"Mach Number Map   t = {t:.4f}")
    ax.set_xlabel("x");  ax.set_ylabel("y")
    ax.set_aspect('auto')

    plt.tight_layout()
    if save:
        fname = f"output/mach_t{t:.4f}.png"
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f"  Saved → {fname}")
    plt.show()


# --------------------------------------------------
# 5. TOTAL ENERGY CONSERVATION
# --------------------------------------------------
def plot_energy_history(history, dx, dy, save=True):
    """
    Plot total energy E_tot = sum(Q[3]) * dx * dy over time.
    Should be conserved (periodic BC, no source).
    """
    times  = [h[0]          for h in history]
    E_tots = [h[1][3].sum() * dx * dy for h in history]

    E0 = E_tots[0] if E_tots[0] != 0 else 1.0
    E_rel = [(e - E_tots[0]) / abs(E0) * 100 for e in E_tots]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    fig.suptitle("Energy Conservation", fontsize=12)

    axes[0].plot(times, E_tots, 'b-o', ms=4)
    axes[0].set_xlabel("t");  axes[0].set_ylabel("Total Energy")
    axes[0].set_title("Absolute Total Energy");  axes[0].grid(True, alpha=0.3)

    axes[1].plot(times, E_rel, 'r-o', ms=4)
    axes[1].set_xlabel("t");  axes[1].set_ylabel("Relative Error (%)")
    axes[1].set_title("Energy Conservation Error");  axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    if save:
        fname = "output/energy_conservation.png"
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f"  Saved → {fname}")
    plt.show()


# --------------------------------------------------
# 6. SAVE ALL PLOTS AT ONCE
# --------------------------------------------------
def save_all(Q, X, Y, dx, dy, t, tag="final"):
    print(f"\n── Saving plots  [{tag}]  t={t:.5f} ──")
    r, u, v, pp, a, M = _primitives(Q)
    print(f"   rho ∈ [{r.min():.4f}, {r.max():.4f}]   "
          f"p ∈ [{pp.min():.4f}, {pp.max():.4f}]   "
          f"Mach_max = {M.max():.3f}")
    plot_contours  (Q, X, Y,     t)
    plot_schlieren (Q, X, Y, dx, dy, t)
    plot_mach      (Q, X, Y,     t)


## 11. Main Driver

In [ ]:
%%writefile main.py
"""
main.py
-------
Main driver for the Fully Compressible 2D Euler Solver.

Runs the following test cases sequentially:
    1. Sod Shock Tube          — primary verification (1-D in 2-D)
    2. 2-D Explosion           — 2-D circular blast wave
    3. Kelvin-Helmholtz        — shear instability (comment in if desired)

Project deliverables produced:
    output/sod_verification_t*.png   — exact vs numerical comparison
    output/schlieren_t*.png          — density-gradient Schlieren images
    output/mach_t*.png               — Mach number maps
    output/contours_t*.png           — full 4-panel field plots
    output/energy_conservation.png   — total energy over time
"""

import numpy as np
import time
import os

os.makedirs("output", exist_ok=True)

import para
import mesh_and_grid as grid
from init_fields import init_sod, init_explosion, init_kh
from fns         import run_simulation
from saving      import (plot_sod_verification, save_all,
                          plot_energy_history)


# ============================================================
#  CASE 1:  SOD SHOCK TUBE  (1-D verification)
# ============================================================
def run_sod():
    print("\n" + "="*60)
    print("  CASE 1 — Sod Shock Tube Verification")
    print("="*60)

    # Thin 2-D slab: 200 x 4 cells, [0,1] x [0,0.02]
    Nx, Ny   = 200, 4
    Lx, Ly   = 1.0, 0.02
    dx, dy   = Lx / Nx, Ly / Ny

    x = (np.arange(Nx) + 0.5) * dx
    y = (np.arange(Ny) + 0.5) * dy
    X, Y = np.meshgrid(x, y, indexing='ij')

    Q = np.zeros((4, Nx, Ny))
    init_sod(Q, X, Y)

    t0 = time.time()
    Q_final, history = run_simulation(Q, dx, dy,
                                      tfinal=0.20,
                                      t_print=0.05,
                                      CFL=0.45)
    print(f"  Wall time: {time.time() - t0:.2f} s")

    # --- Deliverable: Verification plot ---
    plot_sod_verification(Q_final, X, t=0.20)

    # --- Deliverable: Energy conservation ---
    plot_energy_history(history, dx, dy)

    # --- Deliverable: Schlieren + Mach ---
    save_all(Q_final, X, Y, dx, dy, t=0.20, tag="sod_final")

    return Q_final, history


# ============================================================
#  CASE 2:  2-D EXPLOSION  (Schlieren showcase)
# ============================================================
def run_explosion():
    print("\n" + "="*60)
    print("  CASE 2 — 2-D Circular Explosion")
    print("="*60)

    Nx, Ny  = 200, 200
    Lx, Ly  = 1.0, 1.0
    dx, dy  = Lx / Nx, Ly / Ny

    x = (np.arange(Nx) + 0.5) * dx
    y = (np.arange(Ny) + 0.5) * dy
    X, Y = np.meshgrid(x, y, indexing='ij')

    Q = np.zeros((4, Nx, Ny))
    init_explosion(Q, X, Y, r_blast=0.2, p_in=10.0, p_out=1.0)

    t0 = time.time()
    Q_final, history = run_simulation(Q, dx, dy,
                                      tfinal=0.25,
                                      t_print=0.05,
                                      CFL=0.40)
    print(f"  Wall time: {time.time() - t0:.2f} s")

    save_all(Q_final, X, Y, dx, dy, t=0.25, tag="explosion_final")
    plot_energy_history(history, dx, dy)

    return Q_final, history


# ============================================================
#  CASE 3:  KELVIN-HELMHOLTZ  (instability demo)
# ============================================================
def run_kh():
    print("\n" + "="*60)
    print("  CASE 3 — Kelvin-Helmholtz Shear Layer")
    print("="*60)

    Nx, Ny  = 256, 256
    Lx, Ly  = 1.0, 1.0
    dx, dy  = Lx / Nx, Ly / Ny

    x = (np.arange(Nx) + 0.5) * dx
    y = (np.arange(Ny) + 0.5) * dy
    X, Y = np.meshgrid(x, y, indexing='ij')

    Q = np.zeros((4, Nx, Ny))
    init_kh(Q, X, Y, Mach=0.5)

    t0 = time.time()
    Q_final, history = run_simulation(Q, dx, dy,
                                      tfinal=2.0,
                                      t_print=0.4,
                                      CFL=0.40)
    print(f"  Wall time: {time.time() - t0:.2f} s")

    save_all(Q_final, X, Y, dx, dy, t=2.0, tag="kh_final")
    plot_energy_history(history, dx, dy)

    return Q_final, history


# ============================================================
#  ENTRY POINT
# ============================================================
if __name__ == "__main__":
    # Always run Sod first for verification
    Q_sod, hist_sod = run_sod()

    # Then 2-D explosion for visual deliverables
    Q_exp, hist_exp = run_explosion()

    # Optional: KH instability (takes longer)
    # Q_kh, hist_kh = run_kh()

    print("\n✓  All cases complete.  Results saved to ./output/")


## 12. Run Simulation

Run **Case 1 (Sod)** for verification, then **Case 2 (Explosion)** for 2-D deliverables.  
Case 3 (Kelvin-Helmholtz) can be uncommented for extra credit.

In [ ]:
!python3 main.py

## 13. Display Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

output_files = sorted([f for f in os.listdir('output') if f.endswith('.png')])
print(f"Generated {len(output_files)} output files:")
for f in output_files:
    print(" ", f)

for fname in output_files:
    img = mpimg.imread(f'output/{fname}')
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(fname.replace('.png','').replace('_',' '))
    plt.tight_layout()
    plt.show()

## 14. Project Deliverables Summary

| Deliverable | File | Status |
|---|---|---|
| Sod verification (exact vs numerical) | `output/sod_verification_t0.2000.png` | ✅ |
| Schlieren plot `|∇ρ|` | `output/schlieren_t*.png` | ✅ |
| Mach number map | `output/mach_t*.png` | ✅ |
| Full field contours (ρ, u, p, Mach) | `output/contours_t*.png` | ✅ |
| Energy conservation | `output/energy_conservation.png` | ✅ |

### Numerical Methods Used
- **Spatial:** 2nd-order MUSCL with minmod limiter → 2nd-order accuracy, TVD
- **Riemann solver:** HLLC — captures shocks, contacts, rarefactions correctly
- **Time integration:** TVD-RK3 (Shu-Osher) — 3rd-order, total variation diminishing
- **CFL:** Adaptive time step, CFL = 0.45

### How to Switch Test Cases
Edit `main.py` and comment/uncomment `run_sod()`, `run_explosion()`, `run_kh()`.